<h1>SDM<h1>

<h2>imports<h2>

In [1]:
import pandas as pd
import numpy as np
import sqlite3

Logger creëeren

In [2]:
import logging
import os
os.makedirs("logs", exist_ok=True)

logging.basicConfig( 
    filename="logs/sdm_de.log",
    level=logging.INFO, 
    format="%(asctime)s | %(name)s |  %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H-%M-%S",
    force=True
    )

logger = logging.getLogger("DWH")
logger.setLevel(logging.INFO)

<h2>connectie data<h2>

In [3]:
#csv-data inlezen
inventory_levels = pd.read_csv('Data/INVENTORY_LEVELS-data.csv')
product_forecast = pd.read_csv('Data/PRODUCT_FORECAST-data.csv')
sales_target = pd.read_csv('Data/SALES_TARGET-data.csv', encoding='cp1252')

# sqlite data inlezen
crm = sqlite3.connect('Data/CRM-data.sqlite')
go_sales = sqlite3.connect('Data/GO_SALES-data.sqlite')
go_staff = sqlite3.connect('Data/GO_STAFF-data.sqlite')

sdm_conn = sqlite3.connect('Data/sdmdb.db')
sdm_cursor = sdm_conn.cursor()

<h2> databases kunnen lezen <h2>

crm data

In [4]:
age_group = pd.read_sql("SELECT * FROM age_group;",crm)
customer_contact = pd.read_sql("SELECT * FROM customer_contact;",crm)
customer_segment = pd.read_sql("SELECT * FROM customer_segment;",crm)
customer_type = pd.read_sql("SELECT * FROM customer_type;",crm)
sales_territory = pd.read_sql("SELECT * FROM sales_territory;",crm)
customer_store = pd.read_sql("SELECT * FROM customer_store;",crm)
crm_country = pd.read_sql("SELECT * FROM crm_country;",crm)
customer = pd.read_sql("SELECT * FROM customer;",crm)
sales_demographic = pd.read_sql("SELECT * FROM sales_demographic;",crm)
customer_headquarters = pd.read_sql("SELECT * FROM customer_headquarters;",crm)

go_sales data

In [5]:
#go_sales data
order_method = pd.read_sql("SELECT * FROM order_method;",go_sales)
product_line = pd.read_sql("SELECT * FROM product_line;",go_sales)
retailer_site = pd.read_sql("SELECT * FROM retailer_site;",go_sales)
return_reason = pd.read_sql("SELECT * FROM return_reason;",go_sales)
returned_item = pd.read_sql("SELECT * FROM returned_item;",go_sales)
order_details = pd.read_sql("SELECT * FROM order_details;",go_sales)
order_header = pd.read_sql("SELECT * FROM order_header;",go_sales)
product_type = pd.read_sql("SELECT * FROM product_type;",go_sales)
product = pd.read_sql("SELECT * FROM product;",go_sales)
sales_staff = pd.read_sql("SELECT * FROM sales_staff;",go_sales)
country = pd.read_sql("SELECT * FROM country;",go_sales)
sales_branch = pd.read_sql("SELECT * FROM sales_branch;",go_sales)

go_staff data

In [6]:
satisfaction = pd.read_sql("SELECT * FROM satisfaction;",go_staff)
training = pd.read_sql("SELECT * FROM training;",go_staff)
course = pd.read_sql("SELECT * FROM course;",go_staff)
sales_office = pd.read_sql("SELECT * FROM sales_office;",go_staff)
satisfaction_type = pd.read_sql("SELECT * FROM satisfaction_type;",go_staff)
sales_representative = pd.read_sql("SELECT * FROM sales_representative;",go_staff)

In [7]:
#een lijst om in 1 overzicht te kunnen zien in welke databases een lege colom zit

lijstje = [
    ("inventory_levels", inventory_levels),
    ("product_forecast", product_forecast),
    ("sales_target", sales_target),
    ("age_group", age_group),
    ("customer_contact", customer_contact),
    ("customer_segment", customer_segment),
    ("customer_type", customer_type),
    ("sales_territory", sales_territory),
    ("customer_store", customer_store),
    ("crm_country", crm_country),
    ("customer", customer),
    ("sales_demographic", sales_demographic),
    ("customer_headquarters", customer_headquarters),
    ("order_method", order_method),
    ("product_line", product_line),
    ("retailer_site", retailer_site),
    ("return_reason", return_reason),
    ("order_details", order_details),
    ("order_header", order_header),
    ("product_type", product_type),
    ("product", product),
    ("sales_staff", sales_staff),
    ("country", country),
    ("sales_branch", sales_branch),
    ("satisfaction", satisfaction),
    ("training", training),
    ("course", course),
    ("sales_office", sales_office),
    ("satisfaction_type", satisfaction_type),
    ("sales_representative", sales_representative)
]


for naam, df in lijstje:
    for col in df.columns:
        if df[col].isna().any():
            print(f"{naam} - {col}")

customer_contact - EXTENSION
customer_contact - FAX
customer_contact - E_MAIL
customer_store - ADDITION
customer_store - STATE
customer_store - ZIPCODE
customer - CUSTOMER_CODEMR
customer_headquarters - ADDRESS2
customer_headquarters - REGION
customer_headquarters - POSTAL_ZONE
customer_headquarters - PHONE
customer_headquarters - FAX
retailer_site - ADDRESS2
retailer_site - REGION
retailer_site - POSTAL_ZONE
sales_staff - EXTENSION
sales_branch - ADDRESS2
sales_branch - REGION
sales_office - ADDITION
sales_office - REGION
sales_representative - EXTENSION


Nulls vullen

In [8]:
sales_representative['EXTENSION'] = sales_representative['EXTENSION'].fillna(0)

sales_office['ADDITION'] = sales_office['ADDITION'].fillna('-')
sales_office['REGION'] = sales_office['REGION'].fillna(sales_office['REGION'].mode()[0])

sales_branch['ADDRESS2'] = sales_branch['ADDRESS2'].fillna(sales_branch['ADDRESS1'].mode()[0])
sales_branch['REGION'] = sales_branch['REGION'].fillna(sales_branch['REGION'].mode()[0])

sales_staff['EXTENSION'] = sales_staff['EXTENSION'].fillna(sales_staff['EXTENSION'].mode()[0])

retailer_site['ADDRESS2'] = retailer_site['ADDRESS2'].fillna('-')
retailer_site['REGION'] = retailer_site['REGION'].fillna('-')

smollist2 = 'REGION', 'POSTAL_ZONE', 'PHONE', 'FAX'
customer_headquarters['ADDRESS2'] = customer_headquarters['ADDRESS2'].fillna(customer_headquarters['ADDRESS2'].mode()[0])
for i in smollist2:
    customer_headquarters[i] = customer_headquarters[i].fillna(customer_headquarters[i].mode()[0])

customer['CUSTOMER_CODEMR'] = customer['CUSTOMER_CODEMR'].fillna(customer['CUSTOMER_CODEMR'].mode()[0])

customer_store['ADDITION'] = customer_store['ADDITION'].fillna('-')

smollist3 = 'STATE', 'ZIPCODE'
for i in smollist3:
    customer_store[i] = customer_store[i].fillna(customer_store[i].mode()[0])

customer_contact['EXTENSION'] = customer_contact['EXTENSION'].fillna('0')

smollist4 = 'FAX', 'E_MAIL'
for i in smollist4:
    customer_contact[i] = customer_contact[i].fillna(customer_contact[i].mode()[0])


Data typen veranderen

In [9]:
customer_headquarters['CUSTOMER_CODEMR'] = customer_headquarters['CUSTOMER_CODEMR'].astype(int)
customer_headquarters['COUNTRY_CODE'] = customer_headquarters['COUNTRY_CODE'].astype(int)

customer_store['CUSTOMER_SITE_CODE'] = customer_store['CUSTOMER_SITE_CODE'].astype(int)
customer_store['ACTIVE_INDICATOR'] = customer_store['ACTIVE_INDICATOR'].astype(bool)

customer_contact["CUSTOMER_CONTACT_CODE"] = customer_contact["CUSTOMER_CONTACT_CODE"].astype(int)
customer_contact["EXTENSION"] = customer_contact["EXTENSION"].astype(int)

country['COUNTRY_CODE'] = country['COUNTRY_CODE'].astype(int)
sales_office['COUNTRY_CODE'] = sales_office['COUNTRY_CODE'].astype(int)

unieke indexen creëeren

In [10]:
product_forecast['num'] = product_forecast.index
sales_target['num'] = sales_target.index
inventory_levels['INVENTORY_LEVELS_CODE'] = inventory_levels.index
satisfaction['SATISFACTION_CODE'] = satisfaction.index
training['TRAINING_CODE'] = training.index

Renames

In [11]:
customer_headquarters = customer_headquarters.rename(columns={
    'CUSTOMER_CODEMR': 'CUSTOMER_CODE_HQ',
    'COUNTRY_CODE': 'COUNTRY'
})
customer_headquarters = customer_headquarters.rename(columns={
    'ADDRESS1': 'ADRESS1',
    'ADDRESS2': 'ADRESS2'
})

retailer_site = retailer_site.rename(columns={
    'ADDRESS1': 'ADRESS1',
    'ADDRESS2': 'ADRESS2'
})

retailer_site = retailer_site.rename(columns={'COUNTRY_CODE' : 'COUNTRY'})
crm_country = crm_country.rename(columns={"SALES_TERRITORY_CODE": "SALES_TERRITORY"})
return_reason = return_reason.rename(columns={'RETURN_DESCRIPTION_EN': 'RETURN_DESCRIPTION'})
sales_office = sales_office.rename(columns={'COUNTRY_CODE': 'COUNTRY'})
product_type = product_type.rename(columns={'PRODUCT_LINE_CODE': 'PRODUCT_LINE'})
sales_branch = sales_branch.rename(columns={'ADDRESS1': 'ADRESS1',
                                            'ADDRESS2': 'ADRESS2',
                                            'COUNTRY_CODE': 'COUNTRY',
                                            'POSTAL_ZONE': 'POSTAL_CODE'})
product = product.rename(columns={'PRODUCT_TYPE_CODE': 'PRODUCT_TYPE',
                                  'PRODUCT_IMAGE': 'PRODUCT_IMG'})
sales_staff = sales_staff.rename(columns={'SALES_BRANCH_CODE': 'SALES_BRANCH'})
sales_representative = sales_representative.rename(columns={'SALES_OFFICE_CODE': 'SALES_OFFICE',
                                                            'MANAGER_CODE': 'MANAGER'})
customer = customer.rename(columns={'CUSTOMER_TYPE_CODE': 'CUSTOMER_TYPE',
                                    'CUSTOMER_CODEMR': 'CUSTOMER_HEADQUARTERS'})
customer_store = customer_store.rename(columns={'CUSTOMER_CODE':'CUSTOMER', 'COUNTRY_CODE':'CRM_COUNTRY', 'CUSTOMER_SITE_CODE':'CUSTOMER_STORE_CODE'})
customer_contact = customer_contact.rename(columns={'CUSTOMER_SITE_CODE':'CUSTOMER_STORE'})
sales_demographic = sales_demographic.rename(columns={'CUSTOMER_CODEMR':'CUSTOMER_HEADQUARTERS', 'AGE_GROUP_CODE':'AGE_GROUP'})
order_header = order_header.rename(columns={'SALES_BRANCH_CODE':'SALES_BRANCH', 'SALES_STAFF_CODE':'SALES_STAFF', 'RETAILER_SITE_CODE':'RETAILER_SITE', 'ORDER_METHOD_CODE':'ORDER_METHOD'})
order_details = order_details.rename(columns={'ORDER_NUMBER':'ORDER_HEADER', 'PRODUCT_NUMBER':'PRODUCT'})
returned_item = returned_item.rename(columns={'ORDER_DETAIL_CODE':'ORDER_DETAILS', 'RETURN_REASON_CODE':'RETURN_REASON'})
inventory_levels = inventory_levels.rename(columns={'PROTUCT_NUMBER':'PRODUCT'})
product_forecast = product_forecast.rename(columns={'num':'PRODUCT_FORECAST_CODE', 'PRODUCT_NUMBER':'PRODUCT'})
inventory_levels = inventory_levels.rename(columns={'INVENTORY_LEVELS_CODE':'INVENTORY_LEVELS_CODE', 'PRODUCT_NUMBER':'PRODUCT'})
sales_target = sales_target.rename(columns={'PRODUCT_NUMBER':'PRODUCT', 'num':'SALES_TARGET_CODE', 'SALES_STAFF_CODE': 'SALES_STAFF'})
satisfaction = satisfaction.rename(columns={'SALES_REPRESENTATIVE_CODE': 'SALES_REPRESENTATIVE', 'SATISFACTION_TYPE_CODE': 'SATISFACTION_TYPE'})
training = training.rename(columns={'SALES_REPRESENTATIVE_CODE':'SALES_REPRESENTATIVE', 'COURSE_CODE': 'COURSE'})

Validiteit checken

In [12]:
valid_codes = country['COUNTRY_CODE'].unique()

sales_office = sales_office[sales_office['COUNTRY'].isin(valid_codes)]
customer_headquarters = customer_headquarters[customer_headquarters['COUNTRY'].isin(valid_codes)]

SDM creëeren

In [14]:
sql_sdm_script = """
--CRM--

CREATE TABLE age_group (
    AGE_GROUP_CODE INT PRIMARY KEY,
    UPPER_AGE INT,
    LOWER_AGE INT
);

CREATE TABLE customer_segment (
    SEGMENT_CODE INT PRIMARY KEY,
    LANGUAGE VARCHAR(10),
    SEGMENT_NAME VARCHAR(45),
    SEGMENT_DESCRIPTION VARCHAR(100)
);

CREATE TABLE customer_type (
    CUSTOMER_TYPE_CODE INT PRIMARY KEY,
    CUSTOMER_TYPE_EN VARCHAR(45)
);

CREATE TABLE sales_territory (
    SALES_TERRITORY_CODE INT PRIMARY KEY,
    TERRITORY_NAME_EN VARCHAR(45)
);

CREATE TABLE crm_country (
    COUNTRY_CODE INT PRIMARY KEY,
    COUNTRY_EN VARCHAR(45),
    FLAG_IMAGE VARCHAR(100),
    SALES_TERRITORY INT,
    FOREIGN KEY (SALES_TERRITORY) REFERENCES sales_territory(SALES_TERRITORY_CODE)
);

CREATE TABLE customer_headquarters (
    CUSTOMER_CODE_HQ INT PRIMARY KEY,
    CUSTOMER_NAME VARCHAR(100),
    ADRESS1 VARCHAR(100),
    ADRESS2 VARCHAR(100),
    CITY VARCHAR(45),
    REGION VARCHAR(45),
    POSTAL_ZONE VARCHAR(45),
    PHONE VARCHAR(45),
    FAX VARCHAR(45),
    COUNTRY INT,
    SEGMENT_CODE INT,
    FOREIGN KEY (COUNTRY) REFERENCES crm_country(COUNTRY_CODE),
    FOREIGN KEY (SEGMENT_CODE) REFERENCES customer_segment(SEGMENT_CODE)
);

CREATE TABLE sales_demographic(
    DEMOGRAPHIC_CODE INT PRIMARY KEY,
    SALES_PERCENT INT,
    CUSTOMER_HEADQUARTERS INT,
    AGE_GROUP INT,
    FOREIGN KEY (CUSTOMER_HEADQUARTERS) REFERENCES customer_headquarters(CUSTOMER_CODE_HQ),
    FOREIGN KEY (AGE_GROUP) REFERENCES age_group(AGE_GROUP_CODE)
);

CREATE TABLE customer(
    CUSTOMER_CODE INT PRIMARY KEY,
    COMPANY_NAME VARCHAR(45),
    CUSTOMER_TYPE INT,
    CUSTOMER_HEADQUARTERS INT,
    FOREIGN KEY (CUSTOMER_TYPE) REFERENCES customer_type(CUSTOMER_TYPE_CODE),
    FOREIGN KEY (CUSTOMER_HEADQUARTERS) REFERENCES customer_headquarters(CUSTOMER_CODE_HQ)
);

CREATE TABLE customer_store(
    CUSTOMER_STORE_CODE INT PRIMARY KEY,
    STREET VARCHAR(100),
    ADDITION VARCHAR(100),
    CITY VARCHAR(45),
    STATE VARCHAR(45),
    ZIPCODE VARCHAR(45),
    ACTIVE_INDICATOR BOOLEAN,
    CUSTOMER INT,
    CRM_COUNTRY INT,
    FOREIGN KEY (CUSTOMER) REFERENCES customer(CUSTOMER_CODE),
    FOREIGN KEY (CRM_COUNTRY) REFERENCES crm_country(COUNTRY_CODE)
);

CREATE TABLE customer_contact(
    CUSTOMER_CONTACT_CODE INT PRIMARY KEY,
    FIRST_NAME VARCHAR(100),
    LAST_NAME VARCHAR(100),
    JOB_POSITION_EN VARCHAR(45),
    EXTENSION INT,
    FAX VARCHAR(45),
    E_MAIL VARCHAR(100),
    GENDER VARCHAR(100),
    CUSTOMER_STORE INT,
    FOREIGN KEY (CUSTOMER_STORE) REFERENCES customer_store(CUSTOMER_STORE_CODE)
);

--Go Sales--

CREATE TABLE product_line(
    PRODUCT_LINE_CODE INT PRIMARY KEY,
    PRODUCT_LINE_EN VARCHAR(45)
    );

CREATE TABLE product_type(
    PRODUCT_TYPE_CODE INT PRIMARY KEY,
    PRODUCT_TYPE_EN VARCHAR(45),
    PRODUCT_LINE INT,
    FOREIGN KEY (PRODUCT_LINE) REFERENCES product_line(PRODUCT_LINE_CODE)
);

CREATE TABLE product(
    PRODUCT_NUMBER INT PRIMARY KEY,
    DESCRIPTION VARCHAR(100),
    INTRODUCTION_DATE DATE,
    PRODUCTION_COST DECIMAL(10,2),
    MARGIN DECIMAL(10,2),
    PRODUCT_IMG VARCHAR(45),
    LANGUAGE VARCHAR(10),
    PRODUCT_NAME VARCHAR(100),
    PRODUCT_TYPE INT,
    FOREIGN KEY (PRODUCT_TYPE) REFERENCES product_type(PRODUCT_TYPE_CODE)
);

CREATE TABLE go_sales_country(
    COUNTRY_CODE INT PRIMARY KEY,
    COUNTRY VARCHAR(100),
    LANGUAGE VARCHAR(10),
    CURRENCY_NAME VARCHAR(45)
);

CREATE TABLE retailer_site(
    RETAILER_SITE_CODE INT PRIMARY KEY,
    RETAILER_CODE INT,
    ADRESS1 VARCHAR(100),
    ADRESS2 VARCHAR(100),
    CITY VARCHAR(100),
    REGION VARCHAR(100),
    POSTAL_ZONE VARCHAR(45),
    ACTIVE_INDICATOR BOOLEAN,
    COUNTRY INT,
    FOREIGN KEY (COUNTRY) REFERENCES go_sales_country(COUNTRY_CODE)
);

CREATE TABLE sales_branch(
    SALES_BRANCH_CODE INT PRIMARY KEY,
    ADRESS1 VARCHAR(100),
    ADRESS2 VARCHAR(100),
    CITY VARCHAR(100),
    REGION VARCHAR(100),
    POSTAL_CODE VARCHAR(45),
    COUNTRY INT,
    FOREIGN KEY (COUNTRY) REFERENCES go_sales_country(COUNTRY_CODE)
);

CREATE TABLE order_method(
    ORDER_METHOD_CODE INT PRIMARY KEY,
    ORDER_METHOD_EN VARCHAR(45)
);

CREATE TABLE sales_staff(
    SALES_STAFF_CODE INT PRIMARY KEY,
    FIRST_NAME VARCHAR(45),
    LAST_NAME VARCHAR(45),
    POSITION_EN VARCHAR(100),
    WORK_PHONE VARCHAR(45),
    EXTENSION INT,
    FAX VARCHAR(45),
    DATE_HIRED DATE,
    SALES_BRANCH INT,
    FOREIGN KEY (SALES_BRANCH) REFERENCES sales_branch(SALES_BRANCH_CODE)
);

CREATE TABLE order_header(
    ORDER_NUMBER INT PRIMARY KEY,
    RETAILER_NAME VARCHAR(100),
    RETAILER_CONTACT_CODE INT,
    ORDER_DATE DATE,
    RETAILER_SITE INT,
    SALES_STAFF INT,
    SALES_BRANCH INT,
    ORDER_METHOD INT,
    FOREIGN KEY (RETAILER_SITE) REFERENCES retailer_site(RETAILER_SITE_CODE),
    FOREIGN KEY (SALES_STAFF) REFERENCES sales_staff(SALES_STAFF_CODE),
    FOREIGN KEY (SALES_BRANCH) REFERENCES sales_branch(SALES_BRANCH_CODE),
    FOREIGN KEY (ORDER_METHOD) REFERENCES order_method(ORDER_METHOD_CODE)
);

CREATE TABLE return_reason(
    RETURN_REASON_CODE INT PRIMARY KEY,
    RETURN_DESCRIPTION VARCHAR(100)
);

CREATE TABLE order_details(
    ORDER_DETAIL_CODE INT PRIMARY KEY,
    QUANTITY INT,
    UNIT_COST DECIMAL(10,2),
    UNIT_PRICE DECIMAL(10,2),
    UNIT_SALE_PRICE DECIMAL(10,2),
    ORDER_HEADER INT,
    PRODUCT INT,
    FOREIGN KEY (ORDER_HEADER) REFERENCES order_header(ORDER_NUMBER),
    FOREIGN KEY (PRODUCT) REFERENCES product(PRODUCT_NUMBER)
);

CREATE TABLE returned_item(
    RETURN_CODE INT PRIMARY KEY,
    RETURN_DATE DATE,
    RETURN_QUANTITY INT,
    ORDER_DETAILS INT,
    RETURN_REASON INT,
    FOREIGN KEY (ORDER_DETAILS) REFERENCES order_details(ORDER_DETAIL_CODE),
    FOREIGN KEY (RETURN_REASON) REFERENCES return_reason(RETURN_REASON_CODE)
);

--GO STAFF--

CREATE TABLE sales_office (
    SALES_OFFICE_CODE INT PRIMARY KEY,
    STREET VARCHAR(100),
    ADDITION VARCHAR(45),
    CITY VARCHAR(100),
    REGION VARCHAR(100),
    ZIPCODE VARCHAR(45),
    COUNTRY INT,
    FOREIGN KEY (COUNTRY) REFERENCES go_sales_country(COUNTRY_CODE)
);

CREATE TABLE sales_representative(
    SALES_REPRESENTATIVE_CODE INT PRIMARY KEY,
    FIRST_NAME VARCHAR(45),
    LAST_NAME VARCHAR(45),
    POSITION_EN VARCHAR(100),
    WORK_PHONE VARCHAR(45),
    EXTENSION INT,
    FAX VARCHAR(45),
    EMAIL VARCHAR(100),
    DATE_HIRED DATE,
    SALES_OFFICE INT,
    MANAGER INT,
    FOREIGN KEY (SALES_OFFICE) REFERENCES sales_office(SALES_OFFICE_CODE),
    FOREIGN KEY (MANAGER) REFERENCES sales_representative(SALES_REPRESENTATIVE_CODE)
);

CREATE TABLE satisfaction_type(
    SATISFACTION_TYPE_CODE INT PRIMARY KEY,
    SATISFACTION_TYPE_DESCRIPTION VARCHAR(100)
);

CREATE TABLE satisfaction(
    SATISFACTION_CODE INT PRIMARY KEY,
    YEAR INT,
    SALES_REPRESENTATIVE INT,
    SATISFACTION_TYPE INT,
    FOREIGN KEY (SALES_REPRESENTATIVE) REFERENCES sales_representative(SALES_REPRESENTATIVE_CODE),
    FOREIGN KEY (SATISFACTION_TYPE) REFERENCES statisfaction_type(SATISFACTION_TYPE_CODE)
);

CREATE TABLE course(
    COURSE_CODE INT PRIMARY KEY,
    COURSE_DESCRIPTION VARCHAR(100)
);

CREATE TABLE training(
    TRAINING_CODE INT PRIMARY KEY,
    YEAR INT,
    SALES_REPRESENTATIVE INT,
    COURSE INT,
    FOREIGN KEY(SALES_REPRESENTATIVE) REFERENCES sales_representative(SALES_REPRESENTATIVE_CODE),
    FOREIGN KEY(COURSE) REFERENCES course(COURSE_CODE)
);

--inventory levels--

CREATE TABLE inventory_level(
    INVENTORY_LEVELS_CODE INT PRIMARY KEY,
    INVENTORY_YEAR INT,
    INVENTORY_MONTH INT,
    INVENTORY_COUNT INT,
    PRODUCT INT,
    FOREIGN KEY(PRODUCT) REFERENCES product(PRODUCT_NUMBER)
);

--product forecast--

CREATE TABLE product_forecast(
    PRODUCT_FORECAST_CODE INT PRIMARY KEY,
    YEAR INT,
    MONTH INT,
    EXPECTED_VOLUME INT,
    PRODUCT INT,
    FOREIGN KEY(PRODUCT) REFERENCES product(PRODUCT_NUMBER)
);

--sales target--

CREATE TABLE sales_target(
    SALES_TARGET_CODE INT PRIMARY KEY,
    SALES_YEAR INT,
    SALES_PERIOD INT,
    RETAILER_NAME VARCHAR(100),
    RETAILER_CODE INT,
    SALES_TARGET DECIMAL(10,2),
    SALES_STAFF INT,
    PRODUCT INT,
    FOREIGN KEY (SALES_STAFF) REFERENCES sales_staff(SALES_STAFF_CODE),
    FOREIGN KEY (PRODUCT) REFERENCES product(PRODUCT_NUMBER)
);
"""

sdm_cursor.executescript(sql_sdm_script)
sdm_conn.commit()

SDM vullen

In [15]:
sdm_cursor.execute("PRAGMA foreign_keys = OFF;")

# =========================
# 1. ROOT TABLES (NO FK)
# =========================
root = [
    ("age_group", age_group, ["AGE_GROUP_CODE","UPPER_AGE","LOWER_AGE"]),
    ("customer_segment", customer_segment, ["SEGMENT_CODE","LANGUAGE","SEGMENT_NAME","SEGMENT_DESCRIPTION"]),
    ("customer_type", customer_type, ["CUSTOMER_TYPE_CODE","CUSTOMER_TYPE_EN"]),
    ("sales_territory", sales_territory, ["SALES_TERRITORY_CODE","TERRITORY_NAME_EN"]),
    ("order_method", order_method, ["ORDER_METHOD_CODE","ORDER_METHOD_EN"]),
    ("product_line", product_line, ["PRODUCT_LINE_CODE","PRODUCT_LINE_EN"]),
    ("return_reason", return_reason, ["RETURN_REASON_CODE","RETURN_DESCRIPTION"]),
    ("course", course, ["COURSE_CODE","COURSE_DESCRIPTION"]),
    ("satisfaction_type", satisfaction_type, ["SATISFACTION_TYPE_CODE","SATISFACTION_TYPE_DESCRIPTION"]),
    ("go_sales_country", country, ["COUNTRY_CODE", "COUNTRY", "LANGUAGE" , "CURRENCY_NAME"])
]

# =========================
# 2. SECOND LAYER
# =========================
mid = [
    ("go_sales_country", country, ["COUNTRY_CODE","COUNTRY","LANGUAGE","CURRENCY_NAME"]),

    ("crm_country", crm_country, [
        "COUNTRY_CODE","COUNTRY_EN","FLAG_IMAGE","SALES_TERRITORY"
    ]),

    ("product_type", product_type, [
        "PRODUCT_TYPE_CODE","PRODUCT_TYPE_EN","PRODUCT_LINE"
    ]),

    ("sales_branch", sales_branch, [
        "SALES_BRANCH_CODE","ADRESS1","ADRESS2","CITY","REGION","POSTAL_CODE","COUNTRY"
    ]),

    ("sales_office", sales_office, [
        "SALES_OFFICE_CODE","STREET","ADDITION","CITY","REGION","ZIPCODE","COUNTRY"
    ]),
]

# =========================
# 3. THIRD LAYER
# =========================
third = [
    ("customer_headquarters", customer_headquarters, [
        "CUSTOMER_CODE_HQ","CUSTOMER_NAME","ADRESS1","ADRESS2",
        "CITY","REGION","POSTAL_ZONE","PHONE","FAX","COUNTRY","SEGMENT_CODE"
    ]),
    ("retailer_site", retailer_site, [
        "RETAILER_SITE_CODE", "RETAILER_CODE", "ADRESS1", "ADRESS2",
        "CITY", "REGION", "POSTAL_ZONE", "ACTIVE_INDICATOR", "COUNTRY"
    ]),

    ("product", product, [
        "PRODUCT_NUMBER","DESCRIPTION","INTRODUCTION_DATE",
        "PRODUCTION_COST","MARGIN","PRODUCT_IMG",
        "LANGUAGE","PRODUCT_NAME","PRODUCT_TYPE"
    ]),

    ("sales_staff", sales_staff, [
        "SALES_STAFF_CODE","FIRST_NAME","LAST_NAME","POSITION_EN",
        "WORK_PHONE","EXTENSION","FAX","DATE_HIRED","SALES_BRANCH"
    ]),

    ("sales_representative", sales_representative, [
        "SALES_REPRESENTATIVE_CODE","FIRST_NAME","LAST_NAME","POSITION_EN",
        "WORK_PHONE","EXTENSION","FAX","EMAIL","DATE_HIRED",
        "SALES_OFFICE","MANAGER"
    ]),

    ("customer", customer, [
        "CUSTOMER_CODE","COMPANY_NAME","CUSTOMER_TYPE","CUSTOMER_HEADQUARTERS"
    ]),
]

# =========================
# 4. FACT TABLES
# =========================
fact = [
    ("customer_store", customer_store, [
        "CUSTOMER_STORE_CODE","STREET","ADDITION","CITY","STATE",
        "ZIPCODE","ACTIVE_INDICATOR","CUSTOMER","CRM_COUNTRY"
    ]),

    ("customer_contact", customer_contact, [
        "CUSTOMER_CONTACT_CODE","FIRST_NAME","LAST_NAME",
        "JOB_POSITION_EN","EXTENSION","FAX","E_MAIL",
        "GENDER","CUSTOMER_STORE"
    ]),

    ("sales_demographic", sales_demographic, [
        "DEMOGRAPHIC_CODE","SALES_PERCENT",
        "CUSTOMER_HEADQUARTERS","AGE_GROUP"
    ]),

    ("order_header", order_header, [
        "ORDER_NUMBER","RETAILER_NAME","RETAILER_CONTACT_CODE",
        "ORDER_DATE","RETAILER_SITE","SALES_STAFF",
        "SALES_BRANCH","ORDER_METHOD"
    ]),

    ("order_details", order_details, [
        "ORDER_DETAIL_CODE","QUANTITY","UNIT_COST","UNIT_PRICE",
        "UNIT_SALE_PRICE","ORDER_HEADER","PRODUCT"
    ]),

    ("returned_item", returned_item, [
        "RETURN_CODE","RETURN_DATE","RETURN_QUANTITY",
        "ORDER_DETAILS","RETURN_REASON"
    ]),

    ("inventory_level", inventory_levels, [
        "INVENTORY_LEVELS_CODE","INVENTORY_YEAR",
        "INVENTORY_MONTH","INVENTORY_COUNT","PRODUCT"
    ]),

    ("product_forecast", product_forecast, [
        "PRODUCT_FORECAST_CODE","YEAR","MONTH",
        "EXPECTED_VOLUME","PRODUCT"
    ]),

    ("sales_target", sales_target, [
        "SALES_TARGET_CODE","SALES_YEAR","SALES_PERIOD",
        "RETAILER_NAME","RETAILER_CODE","SALES_TARGET",
        "SALES_STAFF","PRODUCT"
    ]),

    ("satisfaction", satisfaction, [
         'SATISFACTION_CODE', 'YEAR', 'SALES_REPRESENTATIVE', 'SATISFACTION_TYPE'
       
    ]),

    ("training", training, [
        "YEAR","SALES_REPRESENTATIVE","COURSE"
    ]),
]

# =========================
# ORDERED EXECUTION
# =========================
all_tables = root + mid + third + fact

for table_name, df, cols in all_tables:

    missing = set(cols) - set(df.columns)
    if missing:
        print(f"❌ {table_name} mist: {missing}")
        continue

    df_clean = df[cols]

    query = f"""
    INSERT OR REPLACE INTO {table_name}
    ({",".join(cols)})
    VALUES ({",".join(["?"] * len(cols))})
    """

    try:
        sdm_cursor.executemany(query, df_clean.values.tolist())
        print(f"✔ {table_name}")
    except Exception as e:
        print(f"❌ {table_name} ERROR: {e}")
        break

sdm_conn.commit()

✔ age_group
✔ customer_segment
✔ customer_type
✔ sales_territory
✔ order_method
✔ product_line
✔ return_reason
✔ course
✔ satisfaction_type
✔ go_sales_country
✔ go_sales_country
✔ crm_country
✔ product_type
✔ sales_branch
✔ sales_office
✔ customer_headquarters
✔ retailer_site
✔ product
✔ sales_staff
✔ sales_representative
✔ customer
✔ customer_store
✔ customer_contact
✔ sales_demographic
✔ order_header
✔ order_details
✔ returned_item
✔ inventory_level
✔ product_forecast
✔ sales_target
✔ satisfaction
✔ training


Reset SDM

In [13]:
def resetSDM():
    """
    Verwijdert alle tabellen uit het SDM
    + logging
    """

    try:
        logger.info("SDM|RESET_SDM_START|Data/SDM.db|||||START")

        sdm_cursor.execute(
            "PRAGMA foreign_keys = OFF;"
        )

        tables = sdm_cursor.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type='table'
        """).fetchall()

        if not tables:
            logger.warning("SDM|ERROR|Data/SDM.db||||FAILED|%s",str(e))

        for (table,) in tables:

            try:
                sdm_cursor.execute(
                    f"DROP TABLE IF EXISTS {table}"
                )

                logger.info(
                "SDM|READ|Data/SDM.db|%s|||SUCCESS",
                table
                )

                print(f"Dropped: {table}")

            except Exception as table_error:

                logger.error(
                f"SDM|ERROR|Data/SDM.db|%s|||FAILED|%s",
                table,
                str(e)
                )

        sdm_conn.commit()

        logger.info("SDM|LOAD_SDM_END|Data/SDM.db|||||END")
        print("SDM volledig gereset (DROP)")

    except Exception as e:

        sdm_conn.rollback()

        logger.error(
            f"logger.error("
            "SDM|ERROR|Data/SDM.db||||FAILED|%s",
            str(e)
            )

        print(f"ERROR: {e}")

    finally:

        sdm_cursor.execute(
            "PRAGMA foreign_keys = ON;"
        )

        logger.info("SDM|RESET_SDM_END|%s|||||END")


# RUN
resetSDM()

Dropped: age_group
Dropped: customer_segment
Dropped: customer_type
Dropped: sales_territory
Dropped: crm_country
Dropped: customer_headquarters
Dropped: sales_demographic
Dropped: customer
Dropped: customer_store
Dropped: customer_contact
Dropped: product_line
Dropped: product_type
Dropped: product
Dropped: go_sales_country
Dropped: retailer_site
Dropped: sales_branch
Dropped: order_method
Dropped: sales_staff
Dropped: order_header
Dropped: return_reason
Dropped: order_details
Dropped: returned_item
Dropped: sales_office
Dropped: sales_representative
Dropped: satisfaction_type
Dropped: satisfaction
Dropped: course
Dropped: training
Dropped: inventory_level
Dropped: product_forecast
Dropped: sales_target
SDM volledig gereset (DROP)
